In [ ]:
import sys
sys.path.append('../../../src/')

import matplotlib.pyplot as plt

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *

import config

In [ ]:
# Test energy is well calculated
def A(c, k):
    return 3*c**4*k**2 - 8*c**2*k**2 - 2*c**2*k + 8*k - 1

def B(c, k):
    return 3*c**2*k - 4*k - 1

def C(c, k):
    return c**2*k - 4*k - 1

def D(c, k):
    return np.sqrt((1 - 2*k) * (2 - c**2))

def L_c(c, k):
    return np.sqrt((2 - c**2) / (1 - 2*k))

s = 1
def E_D_1_3(c, k):
    return (A(c, k) * np.arctan(np.sqrt(k) * L_c(c, k)) - s * np.sqrt(k) * B(c, k) * D(c, k)) / (16 * k**(3/2))

def E_D_2(c, k):
    return -(A(c, k) * np.arctanh(np.sqrt(np.abs(k)) * L_c(c, k))- np.sqrt(np.abs(k)) * B(c, k) * D(c, k)) / (16 * np.abs(k)**(3/2))


In [ ]:
k = -50
pars = {"k":k}
c_0 = np.sqrt(2)/2

P_p_ = lambda c: P_p(c, k)
P_pp_ = lambda c: P_pp(c, k)

c_star, error = newton_1D(c_0, P_p_, P_pp_)
print("c* = ", c_star)

eps = 0.1
c_st = c_star + 3*eps
c_uns = c_star - 3*eps

In [ ]:
L = 400 #80
N = 50_001
Nt = 10_000 #10_000
tf = 10 #1 #15

# Stable

In [ ]:
w_dark_st, x_dark_st = soliton(c_st, k, soliton_type="dark", L=L, N=N, as_func=True)

In [ ]:
E_D_2(c_st, k)

In [ ]:
h = x_dark_st[1] - x_dark_st[0]
w = w_dark_st(x_dark_st)

e_0 = h * np.sum(1/2 * norm(np.gradient(w, h)) + 1/4 * (1 - norm(w))**2 - k/4 * (np.gradient(norm(w), h))**2)
e_0

In [ ]:
w_st = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_st.set_init_condition(w_dark_st)
w_st.solve(b_tol=1e-4)#method="fixed point")

In [ ]:
np.random.seed(67)

def sol_gauss(x, A):
    output = w_dark_st(x) + A*np.exp(-x**2)
    return output

eps_rand = 1e-2
A_rand = np.random.rand(len(x_dark_st)) * eps_rand
A_rand[np.abs(x_dark_st)>=L/2] = eps_rand

init_cond = lambda x: sol_gauss(x, A_rand)

In [ ]:
w_st_gauss_rand = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_st_gauss_rand.set_init_condition(init_cond)
w_st_gauss_rand.solve(b_tol=1e-4)

# Unstable

In [ ]:
w_dark_uns, x_dark_uns = soliton(c_uns, k, soliton_type="dark", L=L, N=N, as_func=True)

In [ ]:
c_zero = 0
w_black_uns, x_black_uns = soliton(c_zero, k, soliton_type="dark", L=L, N=N, as_func=True)

In [ ]:
E_D_2(c_uns, k)

In [ ]:
h = x_dark_uns[1] - x_dark_uns[0]
w = w_dark_uns(x_dark_uns)

e_0 = h * np.sum(1/2 * norm(np.gradient(w, h)) + 1/4 * (1 - norm(w))**2 - k/4 * (np.gradient(norm(w), h))**2)
e_0

In [ ]:
w_uns = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_uns.set_init_condition(w_black_uns)#w_dark_uns)
w_uns.solve(b_tol=1e-4)

In [ ]:
np.random.seed(67)

def sol_gauss_uns(x, A):
    output = w_black_uns(x) + A*np.exp(-x**2) #w_dark_uns(x)
    return output

eps_rand = 1e-2
A_rand = np.random.rand(len(x_black_uns)) * eps_rand
A_rand[np.abs(x_black_uns)>=L/2] = eps_rand
init_cond = lambda x: sol_gauss_uns(x, A_rand)

In [ ]:
w_uns_gauss_rand = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_uns_gauss_rand.set_init_condition(init_cond)
w_uns_gauss_rand.solve(b_tol=1e-4)

# Comparison

In [ ]:
L_theo = 800 #80
N_theo = 100_001

c_zero = 0
w_black_uns, x_black_uns = soliton(c_zero, k, soliton_type="dark", L=L_theo, N=N_theo, as_func=True)

In [ ]:
#def theo_dark_st(x, t):
#    output = w_dark_st(x - c_st*t)
#    return output

def theo_dark_uns(x, t):
    output = w_black_uns(x - c_zero*t) #w_dark_uns(x - c_uns*t)
    return output

In [ ]:
# Stables
#w_st.w_theo = theo_dark_st
#L2_errors_st = w_st.get_error()

#w_st_gauss_rand.w_theo = theo_dark_st
#L2_errors_st_gauss_rand = w_st_gauss_rand.get_error()

# Unstables
w_uns.w_theo = theo_dark_uns
L2_errors_uns = w_uns.get_error()

w_uns_gauss_rand.w_theo = theo_dark_uns
L2_errors_uns_gauss_rand = w_uns_gauss_rand.get_error()


In [ ]:
#cond_st = norm(w_st_gauss_rand.w) != 0
cond_uns = norm(w_uns_gauss_rand.w) != 0

#plt.plot(w_st.t[cond_st[:, 0]], L2_errors_st[cond_st[:, 0]], "-", label="Control DS stable", marker="none")
#plt.plot(w_st_gauss_rand.t[cond_st[:, 0]], L2_errors_st_gauss_rand[cond_st[:, 0]], "-", label="Perturbed DS stable", marker="none")
plt.plot(w_uns.t[cond_uns[:, 0]], L2_errors_uns[cond_uns[:, 0]], "-", label="Control DS unstable", marker="none")

plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$ abs. diff ", rotation='horizontal', ha='right')
#plt.xscale("log")
#plt.yscale("log")
#plt.xlim(-0.1, 0.5)
plt.legend()
plt.show()

In [ ]:
def g(x, A):
    return A * np.exp(-x**2)

norm_g = np.sqrt(w_uns.h) * np.linalg.norm(g(w_uns.x, A_rand), ord=2)

ms = .5

#cond_st = norm(w_st_gauss_rand.w) != 0
cond_uns = norm(w_uns_gauss_rand.w) != 0

#plt.plot(w_st.t[cond_st[:, 0]], L2_errors_st[cond_st[:, 0]], "-", label="Control DS stable", marker="none")
#plt.plot(w_st_gauss_rand.t[cond_st[:, 0]], L2_errors_st_gauss_rand[cond_st[:, 0]], "-", label="Perturbed DS stable", marker="none")
plt.plot(w_uns_gauss_rand.t[cond_uns[:, 0]], L2_errors_uns_gauss_rand[cond_uns[:, 0]]/norm_g, "-", label="Random Perturbed DS unstable", marker="none")

plt.title(r"$c = 0$")
plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$ rel. diff", rotation='horizontal', ha='right')
#plt.xscale("log")
plt.yscale("log")
#plt.xlim(-0.1, 0.5)
plt.legend()
plt.show()

In [ ]:
from scipy.optimize import curve_fit

Ti = 0.3 #1.5 
Tf = 0.8 # 2.4

t = w_uns_gauss_rand.t
dt = w_uns_gauss_rand.dt

cond_t = np.logical_and(Ti<t, t<Tf)

# Exponential

def line_func(t, C, l):
    return l*t + C#C * np.exp(l*t)

pars_uns_rand, other = curve_fit(line_func,  t[cond_t],  np.log(L2_errors_uns_gauss_rand[cond_t]))
C_rand_uns, l_rand_uns = pars_uns_rand


In [ ]:
plt.plot(t[cond_t], L2_errors_uns_gauss_rand[cond_t]/norm_g, "-", label="Random Perturbed DS unstable", marker="none")

plt.plot(t[cond_t], np.exp(line_func(t[cond_t], C_rand_uns, l_rand_uns))/norm_g, marker="none",
         label=r"Exp rand. uns. $\lambda=${}".format(round(l_rand_uns, 4)))

plt.title(r"$c = 0$")
plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$ rel. diff", rotation='horizontal', ha='right')
#plt.xscale("log")
plt.yscale("log")
#plt.xlim(-0.1, 0.5)
plt.legend()
plt.show()

In [ ]:
def g(x, A):
    return A * np.exp(-x**2)

In [ ]:
ms = .5
norm_g = np.sqrt(w_uns.h) * np.linalg.norm(g(x_dark_uns, A_rand), ord=2)

cond_st = norm(w_st_gauss_rand.w) != 0
cond_uns = norm(w_uns_gauss_rand.w) != 0

plt.plot(w_st.t[cond_st[:, 0]], L2_errors_st[cond_st[:, 0]]/norm_g, "-", label="Control DS stable", marker="none")
plt.plot(w_st_gauss_rand.t[cond_st[:, 0]], L2_errors_st_gauss_rand[cond_st[:, 0]]/norm_g, "-", label="Perturbed DS stable", marker="none")
plt.plot(w_uns.t[cond_uns[:, 0]], L2_errors_uns[cond_uns[:, 0]]/norm_g, "-", label="Control DS unstable", marker="none")
plt.plot(w_uns_gauss_rand.t[cond_uns[:, 0]], L2_errors_uns_gauss_rand[cond_uns[:, 0]]/norm_g, "-", label="Perturbed DS unstable", marker="none")

plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$-diff rel", rotation='horizontal', ha='right')
#plt.xscale("log")
plt.yscale("log")
#plt.xlim(-0.1, 0.5)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))


plt.plot(x_dark_st, norm(w_st.w[2729]), ls="-", marker="none", label="Control stable DS")
plt.plot(x_dark_st, norm(w_st_gauss_rand.w[2729]), ls="--", marker="none", label="Perturbed stable DS")

plt.plot(x_dark_st, norm(w_uns.w[2729]), ls="-", marker="none", label="Control unstable DS")
plt.plot(x_dark_st, norm(w_uns_gauss_rand.w[2729]), ls="--", marker="none", label="Perturbed unstable DS")

plt.legend(fontsize=7)
plt.xlim(-15, 30)
plt.show()

In [ ]:
x = np.linspace(0, 20, 1000)
plt.plot(x, x**3, "-", label=r"$x^3$", marker="none")
plt.xlim(0, 20)